In [8]:
!pip install torcheeg scipy==1.12.0 numpy==1.26.4 torch_geometric

In [9]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
phhasian0710_seed_iv_path = kagglehub.dataset_download('phhasian0710/seed-iv')

print('Data source import complete.')


Using Colab cache for faster access to the 'seed-iv' dataset.
Data source import complete.


In [10]:
import torch
import torch.nn as nn
import torch.nn.utils as utils
from torch.utils.data import DataLoader, Subset,WeightedRandomSampler
from torcheeg.models import CCNN
from torcheeg import transforms
from torcheeg.transforms import ToGrid
from torcheeg.datasets import SEEDIVDataset,SEEDIVFeatureDataset
from torcheeg.datasets.constants import SEED_IV_CHANNEL_LOCATION_DICT
# from torcheeg.transforms import ToG
from torcheeg.datasets.constants import SEED_IV_ADJACENCY_MATRIX
from torcheeg.models import DGCNN
from torch.optim.lr_scheduler import CosineAnnealingLR
# --- THE MAIN SUBJECT LOOP ---

import torch_geometric.loader as geom_loader # Special loader for graphs
import copy
import scipy.signal as signal
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Function
import numpy as np
import os

In [11]:
# 1. Setup Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [12]:
def map_emotions(y):
    if y == 1 or y == 2:  # Sad or Fear
        return 0
    if y == 3:  # Happy
        return 1
    return -1 # Neutral

In [13]:
# Set a fixed random seed for reproducibility across different libraries.
def set_seed(seed_value=42):
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
        # for GPU
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

In [15]:
dataset = SEEDIVFeatureDataset(
    io_path='./tmp_out/seed_iv_features',
    root_path='/kaggle/input/seed-iv/eeg_feature_smooth',
    feature=['de_LDS'],
    num_worker=0,
    offline_transform=transforms.Compose([
        transforms.To2d(),
        transforms.Lambda(lambda x: torch.tensor(x).float())]),
    label_transform=transforms.Select('emotion'),
)

[2026-03-06 14:34:03] INFO (torcheeg/MainThread) 🔍 | Processing EEG data. Processed EEG data has been cached to ./tmp_out/seed_iv_features.
INFO:torcheeg:🔍 | Processing EEG data. Processed EEG data has been cached to ./tmp_out/seed_iv_features.
[2026-03-06 14:34:03] INFO (torcheeg/MainThread) ⏳ | Monitoring the detailed processing of a record for debugging. The processing of other records will only be reported in percentage to keep it clean.
INFO:torcheeg:⏳ | Monitoring the detailed processing of a record for debugging. The processing of other records will only be reported in percentage to keep it clean.
[PROCESS]:   0%|          | 0/45 [00:00<?, ?it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4_20151111.mat]: 0it [00:00, ?it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4_20151111.mat]: 1it [00:00,  3.65it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4_20151111.mat]: 15it [00:00, 49.52it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4_20151111.mat]: 

In [16]:
# This DataFrame contains columns like: subject_id, trial_id, emotion, etc.
meta_info = dataset.info
all_subjects = sorted(meta_info['subject_id'].unique())
print(f"Subjects Found: {all_subjects}")

Subjects Found: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]


In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GCNLayer(nn.Module):
    """Basic Graph Convolution Layer."""
    def __init__(self, in_features, out_features):
        super(GCNLayer, self).__init__()
        self.weight = nn.Parameter(torch.FloatTensor(in_features, out_features))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x, adj):
        # x: (B, N, Fin), adj: (B, N, N)
        support = torch.matmul(x, self.weight)
        output = torch.matmul(adj, support)
        return F.relu(output)

class MultiGraphicLayerConstruction(nn.Module):
    """Transformer-based adaptive adjacency matrix construction."""
    def __init__(self, feature_dim, num_heads=2, k=10):
        super(MultiGraphicLayerConstruction, self).__init__()
        self.k = k
        self.num_heads = num_heads
        self.norm1 = nn.LayerNorm(feature_dim)
        self.multihead_attn = nn.MultiheadAttention(embed_dim=feature_dim, num_heads=num_heads, batch_first=True)

        self.norm2 = nn.LayerNorm(feature_dim)
        self.mlp = nn.Sequential(
            nn.Linear(feature_dim, feature_dim * 2),
            nn.ReLU(),
            nn.Linear(feature_dim * 2, feature_dim)
        )

       # داخل كلاس MultiGraphicLayerConstruction
    def forward(self, x):
        B, N, D = x.shape
        x_norm = self.norm1(x)

        # 1. Scaled Dot-Product
        scaling = D ** 0.5
        _, attn_weights = self.multihead_attn(x_norm, x_norm, x_norm)

        attn_out, _ = self.multihead_attn(x_norm, x_norm, x_norm)
        x2 = self.norm2(x + attn_out)
        g_feature = self.mlp(x2) + x2

        # 2. Scaling الـ Matrix multiplication قبل الـ Softmax
        s_matrix = torch.matmul(g_feature, g_feature.transpose(-1, -2)) / scaling

        # 3. Adjacency Matrix
        adj = F.softmax(attn_weights + s_matrix, dim=-1)

        # 4. Top-K Sparsity (مهم جداً للـ Stability)
        # تأكد إن الـ K مش كبيرة أوي (10 مناسبة جداً لـ 62 قناة)
        topk_val, _ = torch.topk(adj, self.k, dim=-1)
        mask = (adj >= topk_val[:, :, -1].unsqueeze(-1)).float()
        adj = adj * mask

        return adj, g_feature

class BFENet_SingleBand(nn.Module):
    """Implementation of BFE-Net for one frequency band."""
    def __init__(self, num_channels=62):
        super(BFENet_SingleBand, self).__init__()

        # CNN Layer: Processes single-channel features independently
        # Input (B, 1, 1) -> Paper implies processing the DE feature dimension
        # Since input is (B, 62, 5), for a single band it's (B, 62, 1)
        self.cnn1 = nn.Conv1d(1, 64, kernel_size=1)
        self.cnn2 = nn.Conv1d(64, 128, kernel_size=1)
        self.cnn3 = nn.Conv1d(128, 256, kernel_size=1)

        self.graph_const = MultiGraphicLayerConstruction(feature_dim=256)
        self.gcn = GCNLayer(256, 128)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        # x: (B, 62, 1) -> treat 62 as nodes, 1 as feature
        B, N, _ = x.shape
        x = x.transpose(1, 2) # (B, 1, 62)

        # CNN Feature Extraction
        x = F.relu(self.cnn1(x))
        x = F.relu(self.cnn2(x))
        x = F.relu(self.cnn3(x)) # (B, 256, 62)

        x = x.transpose(1, 2) # (B, 62, 256) -> (B, Nodes, Features)

        # Adaptive Graph Construction
        adj, g_features = self.graph_const(x)

        # Graph Convolution
        x_gcn = self.gcn(g_features, adj) # (B, 62, 128)

        # Flatten for fusion
        x_flat = x_gcn.reshape(B, -1)
        return x_flat

class BFENet_Full(nn.Module):
    """The full model that fuses 5 frequency bands."""
    def __init__(self, num_classes=3): # 3 for SEED, 4 for SEED-IV
        super(BFENet_Full, self).__init__()
        self.bands = nn.ModuleList([BFENet_SingleBand() for _ in range(5)])

        # Final classification layer
        # Input: 5 bands * (62 nodes * 128 gcn features)
        self.classifier = nn.Sequential(
            nn.Linear(5 * 62 * 128, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        # x shape: (B, 62, 5)
        band_features = []
        for i in range(5):
            band_data = x[:, :, i].unsqueeze(-1) # (B, 62, 1)
            band_out = self.bands[i](band_data)
            band_features.append(band_out)

        # Fusion
        merged = torch.cat(band_features, dim=1)
        logits = self.classifier(merged)
        return logits



In [27]:
# Parameters
EPOCHS = 200
BATCH_SIZE = 64
LR = 1e-5
PATIENCE_LR = 3
REDUCE_FACTOR = 0.5
PATIENCE_ES = 25
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1

In [ ]:
from sklearn.model_selection import train_test_split

# Create a directory to save the best models
if not os.path.exists('saved_models'):
    os.makedirs('saved_models')


print("Start Subject-INDEPENDENT (LOSO) Evaluation...")

meta_info = dataset.info
all_subjects = sorted(meta_info['subject_id'].unique())

subject_results = {}

for test_subject in all_subjects:
    print(f"\n{'='*30} Testing on Subject {test_subject} {'='*30}")

    # =========================
    # Split Train / Test (LOSO)
    # =========================
    train_indices = meta_info[meta_info['subject_id'] != test_subject].index.tolist()
    test_indices  = meta_info[meta_info['subject_id'] == test_subject].index.tolist()

    print(f"Training samples: {len(train_indices)} | Testing samples: {len(test_indices)}")

    train_set = Subset(dataset, train_indices)
    test_set  = Subset(dataset, test_indices)

    # =========================
    # Class balancing (train only)
    # =========================
    y_train = meta_info.iloc[train_indices]['emotion'].values
    class_counts = np.bincount(y_train)
    class_weights = 1. / (class_counts + 1e-6)
    sample_weights = [class_weights[y] for y in y_train]

    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )

    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, sampler=sampler)
    test_loader  = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

    # =========================
    # Model
    # =========================
    model = BFENet_Full(num_classes=4).to(device)

    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS, eta_min=1e-6
    )

    best_val_acc = 0.0
    counter = 0

    # =========================
    # Training Loop
    # =========================
    for epoch in range(EPOCHS):
        model.train()
        correct = 0
        total = 0
        train_loss = 0

        for X, y in train_loader:
            X = X.to(device)
            y = y.to(device).long()

            # (B,1,62,5) -> (B,62,5)
            if len(X.shape) == 4:
                X = X.squeeze(1)

            # -------- Standardization --------
            mean = X.mean(dim=(1,2), keepdim=True)
            std  = X.std(dim=(1,2), keepdim=True) + 1e-6
            X = (X - mean) / std

            optimizer.zero_grad()
            outputs = model(X)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += y.size(0)
            correct += (predicted == y).sum().item()

        train_acc = 100 * correct / total
        avg_train_loss = train_loss / len(train_loader)

        # =========================
        # Validation (test subject)
        # =========================
        model.eval()
        val_correct = 0
        val_total = 0
        val_loss = 0

        with torch.no_grad():
            for X, y in test_loader:
                X = X.to(device)
                y = y.to(device).long()

                if len(X.shape) == 4:
                    X = X.squeeze(1)

                mean = X.mean(dim=(1,2), keepdim=True)
                std  = X.std(dim=(1,2), keepdim=True) + 1e-6
                X = (X - mean) / std

                outputs = model(X)
                val_loss += criterion(outputs, y).item()

                _, predicted = torch.max(outputs, 1)
                val_total += y.size(0)
                val_correct += (predicted == y).sum().item()

        val_acc = 100 * val_correct / val_total
        avg_val_loss = val_loss / len(test_loader)

        scheduler.step()

        print(f"Epoch {epoch+1:02d} | "
              f"Train Loss: {avg_train_loss:.4f} Acc: {train_acc:.2f}% | "
              f"Val Loss: {avg_val_loss:.4f} Acc: {val_acc:.2f}%")

        # =========================
        # Early Stopping
        # =========================
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            counter = 0
            torch.save(model.state_dict(), f"saved_models/subject_{test_subject}_best.pth")
        else:
            counter += 1
            if counter >= PATIENCE_ES:
                print(f"Early stopping at epoch {epoch}")
                break

    print(f"Subject {test_subject} Best Accuracy: {best_val_acc:.2f}%")
    subject_results[test_subject] = best_val_acc


# ================= FINAL REPORT =================
print("\n" + "="*40)
print("FINAL SUBJECT-INDEPENDENT RESULTS")
print("="*40)

all_accuracies = list(subject_results.values())

for sub_id in sorted(subject_results.keys()):
    print(f"Subject {sub_id}: {subject_results[sub_id]:.2f}%")

print(f"\nAverage Accuracy: {np.mean(all_accuracies):.2f}%")
print(f"Std: {np.std(all_accuracies):.2f}")
print("="*40)

Start Subject-INDEPENDENT (LOSO) Evaluation...

============================== Testing on Subject 1 ==============================
Training samples: 35070 | Testing samples: 2505
Epoch 01 | Train Loss: 1.3018 Acc: 39.01% | Val Loss: 1.3707 Acc: 41.44%
Epoch 02 | Train Loss: 0.9417 Acc: 69.16% | Val Loss: 1.2178 Acc: 46.39%
Epoch 03 | Train Loss: 0.7197 Acc: 83.31% | Val Loss: 1.2152 Acc: 43.55%
Epoch 04 | Train Loss: 0.5967 Acc: 90.56% | Val Loss: 1.4360 Acc: 30.90%
Epoch 05 | Train Loss: 0.5214 Acc: 95.06% | Val Loss: 1.7253 Acc: 31.14%
Epoch 06 | Train Loss: 0.4693 Acc: 97.83% | Val Loss: 1.7683 Acc: 30.38%
Epoch 07 | Train Loss: 0.4372 Acc: 98.92% | Val Loss: 1.9882 Acc: 28.54%
Epoch 08 | Train Loss: 0.4150 Acc: 99.45% | Val Loss: 1.9166 Acc: 32.18%
Epoch 09 | Train Loss: 0.3992 Acc: 99.67% | Val Loss: 2.1222 Acc: 32.02%
Epoch 10 | Train Loss: 0.3891 Acc: 99.79% | Val Loss: 2.1810 Acc: 31.18%
Epoch 11 | Train Loss: 0.3819 Acc: 99.86% | Val Loss: 2.2550 Acc: 30.42%
Epoch 12 | Train L